In [4]:
import numpy as np
import scipy.signal as signal
import soundfile as sf
from scipy.io import wavfile


In [5]:

def generate_filtered_pink_pulse(duration_ms=2000, fs=48000, band=(8000, 16000), amplitude=0.25):
    """
    Generate filtered pink noise pulse at final output amplitude.
    No additional scaling needed in Bonsai.
    """
    # Duration in samples
    duration = int((duration_ms / 1000) * fs)
    
    # Generate pink noise using Voss method approximation
    def pink_noise(n_samples):
        white = np.random.randn(n_samples)
        b, a = signal.butter(1, 0.01, btype='low')  # crude pink approximation
        return signal.lfilter(b, a, white)

    noise = pink_noise(duration * 2)  # generate more to allow for filtering edge artifacts

    # Bandpass filter design
    nyq = fs / 2
    b, a = signal.butter(4, [band[0]/nyq, band[1]/nyq], btype='band')
    filtered = signal.lfilter(b, a, noise)

    # Truncate to exact duration
    filtered = filtered[:duration]

    # Apply 16 ms sinusoidal ramp
    ramp_samples = int(0.016 * fs)
    ramp = np.sin(np.linspace(0, np.pi/2, ramp_samples))**2  # raised cosine ramp
    envelope = np.ones_like(filtered)
    envelope[:ramp_samples] *= ramp
    envelope[-ramp_samples:] *= ramp[::-1]
    shaped = filtered * envelope

    # Normalize and set to final amplitude (ready to use)
    shaped /= np.max(np.abs(shaped))  # normalize to 1.0
    shaped *= amplitude  # set final amplitude - no scaling needed in Bonsai

    return shaped, fs


In [8]:
def generate_white_noise(duration_s=60, fs=44100, amplitude=0.1):
    """
    Generate white noise at final output amplitude.
    No additional scaling needed in Bonsai.
    """
    noise = np.random.randn(int(duration_s * fs))
    noise /= np.max(np.abs(noise))
    noise *= amplitude  # set final amplitude - no scaling needed in Bonsai
    return noise, fs



In [4]:
def generate_go_cue_wav(output_filename="go_cue.wav"):
    """
    Generate a WAV file with a 10 kHz pure tone for 0.1 seconds.
    
    Parameters:
    - output_filename: Name of the WAV file to create
    
    This generates the auditory Go cue as described in the paper:
    - Frequency: 10 kHz
    - Duration: 0.1 seconds
    """
    # Audio parameters
    frequency = 10000  # 10 kHz
    duration = 0.1     # 0.1 seconds
    sample_rate = 44100  # Standard audio sample rate
    amplitude = 0.9    # Volume (0.0 to 1.0) - using 0.9 for a strong signal
    
    # Generate time array
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    
    # Generate tone (sine wave)
    tone = amplitude * np.sin(2 * np.pi * frequency * t)
    
    # Ensure the audio data is in the correct format for WAV files
    # Convert to 16-bit PCM
    tone_int16 = (tone * 32767).astype(np.int16)
    
    # Save as WAV file
    wavfile.write(output_filename, sample_rate, tone_int16)
    print(f"Go cue audio saved to {output_filename}")

# Generate the WAV file
if __name__ == "__main__":
    generate_go_cue_wav()


Go cue audio saved to go_cue.wav


In [9]:
if __name__ == "__main__":
    # Generate filtered pink pulse (ready to use - no scaling needed)
    pulse, fs_pulse = generate_filtered_pink_pulse(
        duration_ms=2000, 
        fs=48000, 
        amplitude=0.25  # Final amplitude for stimulus presentation
    )
    sf.write("filtered_pink_pulse.wav", pulse, fs_pulse)
    print("Filtered pink pulse saved")

    # Generate background white noise (ready to use - no scaling needed)
    bg_noise, fs_noise = generate_white_noise(
        duration_s=7200,
        fs=44100,
        amplitude=0.1  # Final amplitude for background noise
    )
    sf.write("background_white_noise.wav", bg_noise, fs_noise)
    print("Background white noise saved")

Filtered pink pulse saved
Background white noise saved
